# Easing Curves Visualization - Light Controller v2.3

Interactive visualization of all PWM/DAC easing modes including custom functions.

## Documentation Links

- **[PWM & Ramp Control Guide](PWM_RAMP_CONTROL.md)** - Complete easing documentation
- **[F Mode Custom Functions](F_MODE_CUSTOM_FUNCTIONS.md)** - Custom function guide (heartbeat, bounce, etc.)
- **[Mock Arduino Guide](MOCK_ARDUINO_GUIDE.md)** - Test protocols without hardware
- **[Protocol Syntax Reference](PROTOCOL_SYNTAX_REFERENCE.md)** - Complete syntax guide
- **[README](../README.md)** - Main project documentation

## Value Formats (v2.3)

| Format | Range | Description |
|--------|-------|-------------|
| **Integer 0/1** | Binary ON/OFF | `0` = OFF, `1` = HIGH (max value: 255 or 4095) |
| **Float 0.0-1.0** | Normalized | Scales to channel max (e.g., `0.5` = 50% = 127 or 2047) |
| **Integer 0-255** | 8-bit PWM | Direct PWM value for Arduino pins |
| **Integer 0-4095** | 12-bit DAC | Direct DAC value for MCP4728/Native DAC |

> ⚠️ **Important:** Integer `1` and float `1.0` are both treated as **HIGH** (max value), not as literal values!

## Available Easing Modes

| Mode | Format | Description |
|------|--------|-------------|
| **L** | `(L:start,end,duration)` | Linear - constant speed |
| **I** | `(I:start,end,duration)` | Ease-In - slow start |
| **O** | `(O:start,end,duration)` | Ease-Out - fast start |
| **C** | `(C:start,end,duration)` | Cosine - full S-curve |
| **X** | `(X:duration\|t_start,t_end)` | Custom t range |
| **F** | `(F:func_name,duration)` | Custom function |

In [215]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

In [ ]:
def f(t):
    """Core easing function: f(t) = (1 - cos(π*t)) / 2"""
    return (1 - np.cos(np.pi * t)) / 2

def calculate_eased_pwm(progress, start_pwm, end_pwm, t_start, t_end):
    """
    Calculate PWM value with easing.
    
    Args:
        progress: 0 to 1 (elapsed time proportion)
        start_pwm: Starting PWM (0-255 for PWM, 0-4095 for DAC)
        end_pwm: Ending PWM (0-255 for PWM, 0-4095 for DAC)
        t_start: Start of t range (0-2)
        t_end: End of t range (0-2)
    """
    # Map progress to t range
    t = t_start + progress * (t_end - t_start)
    
    # Calculate f(t) at endpoints and current
    f_start = f(t_start)
    f_end = f(t_end)
    f_current = f(t)
    
    # Normalize to [0, 1]
    if abs(f_end - f_start) < 0.0001:
        eased_progress = progress
    else:
        eased_progress = (f_current - f_start) / (f_end - f_start)
    
    # Calculate output value
    output = start_pwm + eased_progress * (end_pwm - start_pwm)
    return np.clip(output, 0, max(start_pwm, end_pwm, 4095))

# Channel type constants (matching Arduino firmware)
OUTPUT_TYPE_PWM = 'P'      # 8-bit PWM (0-255)
OUTPUT_TYPE_DAC = 'D'      # 12-bit Native DAC (0-4095)
OUTPUT_TYPE_MCP4728 = 'M'  # 12-bit MCP4728 DAC (0-4095)
OUTPUT_TYPE_BINARY = 'B'   # Binary (0 or 1)

def get_max_value(channel_type='P'):
    """Get maximum value for a channel type."""
    if channel_type in ('D', 'M'):  # DAC or MCP4728
        return 4095
    elif channel_type == 'B':  # Binary
        return 1
    else:  # PWM
        return 255

def convert_status_value(value, channel_type='P'):
    """
    Convert a status value to output value based on channel type.
    
    This matches the v2.3 implementation in lcfunc.py:
    - Integer 0/1: Binary (0 = OFF, 1 = HIGH/max)
    - Float 0.0-1.0: Normalized (scales to max)
    - Integer 2-255: 8-bit PWM value
    - Integer 256-4095: 12-bit DAC value
    """
    max_val = get_max_value(channel_type)
    
    # Check if it's a float with decimal
    has_decimal = isinstance(value, float) and (value % 1 != 0)
    
    # Normalized float (0.0-1.0) → scale to max
    if has_decimal and 0.0 <= value <= 1.0:
        return int(round(value * max_val))
    
    # Integer handling
    int_value = int(value)
    
    # Integer 0 or 1 → binary (OFF/ON)
    if int_value == 0:
        return 0
    if int_value == 1:
        return max_val  # 1 means HIGH (255 for PWM, 4095 for DAC)
    
    # Other integers → pass through (with capping)
    return min(int_value, max_val)

print("Value conversion examples:")
print("=" * 50)
for ch_type, type_name in [('P', 'PWM (8-bit)'), ('M', 'DAC (12-bit)')]:
    print(f"\n{type_name}:")
    for val in [0, 1, 0.0, 0.5, 1.0, 128, 2048]:
        result = convert_status_value(val, ch_type)
        print(f"  {val!r:>6} → {result}")

## 1. The Full Cosine Curve

First, let's visualize the entire cosine-based easing curve over $t \in [0, 2]$:

In [217]:
t_full = np.linspace(0, 2, 500)
f_full = f(t_full)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=t_full, y=f_full,
    mode='lines',
    name='f(t) = (1 - cos(πt)) / 2',
    line=dict(color='blue', width=3)
))

# Add key points
key_t = [0, 0.5, 1, 1.5, 2]
key_f = [f(t) for t in key_t]
fig.add_trace(go.Scatter(
    x=key_t, y=key_f,
    mode='markers+text',
    name='Key Points',
    marker=dict(size=12, color='red'),
    text=[f't={t}\nf={v:.2f}' for t, v in zip(key_t, key_f)],
    textposition='top center'
))

# Shaded regions with CORRECT labels
fig.add_vrect(x0=0, x1=0.5, fillcolor='green', opacity=0.15, 
              annotation_text='Ease-In ↑ (t: 0→0.5)', annotation_position='top left')
fig.add_vrect(x0=0.5, x1=1, fillcolor='orange', opacity=0.15,
              annotation_text='Ease-Out ↑ (t: 0.5→1)', annotation_position='top left')
fig.add_vrect(x0=1, x1=1.5, fillcolor='green', opacity=0.1,
              annotation_text='Ease-In ↓ (t: 1→1.5)', annotation_position='bottom right')
fig.add_vrect(x0=1.5, x1=2, fillcolor='orange', opacity=0.1,
              annotation_text='Ease-Out ↓ (t: 1.5→2)', annotation_position='bottom right')

fig.update_layout(
    title='Complete Cosine Easing Curve: f(t) = (1 - cos(πt)) / 2',
    xaxis_title='t (range parameter)',
    yaxis_title='f(t) (normalized progress 0-1)',
    height=500,
    showlegend=True
)

fig.show()

## 2. Easing Modes Comparison

The RAMP command supports these easing modes, based on the cosine curve:

| Mode | Code | t Range (ascending) | t Range (descending) | Description |
|------|------|---------------------|----------------------|-------------|
| **L** (Linear) | `L` | N/A | N/A | Constant speed |
| **C** (Cosine) | `C` | [0, 1] | [1, 2] | Full S-curve (ease-in-out) |
| **I** (Ease-In) | `I` | [0, 0.5] | [1, 1.5] | Slow start, accelerating |
| **O** (Ease-Out) | `O` | [0.5, 1] | [1.5, 2] | Fast start, decelerating |
| **X** (Custom) | `X` | [t_start, t_end] | any | User-defined curve portion |
| **F** (Function) | `F` | N/A | N/A | Custom function from custom_easing.h |

**Key Insight:** The output range (0-255 for PWM, 0-4095 for DAC) scales ANY curve portion to match start→end values. The t range only affects the *shape* of the transition.

**Value Examples:** `RAMP:(I:0,1,5000)` ramps 0→255 (PWM) or 0→4095 (DAC) depending on channel type.

In [218]:
# Time progress from 0 to 1
progress = np.linspace(0, 1, 200)

# Calculate eased values for different modes (CORRECT t ranges)
linear = progress  # L mode
ease_in = np.array([calculate_eased_pwm(p, 0, 1, 0, 0.5) for p in progress])    # I mode (t: 0→0.5) - slow start
ease_out = np.array([calculate_eased_pwm(p, 0, 1, 0.5, 1) for p in progress])   # O mode (t: 0.5→1) - slow end  
cosine_full = np.array([calculate_eased_pwm(p, 0, 1, 0, 1) for p in progress])  # C mode (t: 0→1) - S-curve

fig = go.Figure()

fig.add_trace(go.Scatter(x=progress, y=linear, name='L (Linear)', 
                         line=dict(color='gray', width=2, dash='dash')))
fig.add_trace(go.Scatter(x=progress, y=ease_in, name='I (Ease-In, t: 0→0.5)', 
                         line=dict(color='green', width=3)))
fig.add_trace(go.Scatter(x=progress, y=ease_out, name='O (Ease-Out, t: 0.5→1)', 
                         line=dict(color='orange', width=3)))
fig.add_trace(go.Scatter(x=progress, y=cosine_full, name='C (Full S-curve, t: 0→1)', 
                         line=dict(color='purple', width=3)))

fig.update_layout(
    title='Easing Modes Comparison (PWM 0→255 ascending transition)',
    xaxis_title="Time Progress",
    yaxis_title='Normalized Output (0-1)',
    height=500,
    legend=dict(x=0.02, y=0.98)
)

fig.show()

## 3. PWM Intensity Over Time

Let's see how the actual PWM values change during a 5-second ramp from 0 to 255:

In [219]:
# 5 second ramp, sampled every 50ms
duration_ms = 5000
time_ms = np.linspace(0, duration_ms, 100)
progress = time_ms / duration_ms

# Calculate PWM for each mode with CORRECT t ranges for ASCENDING (0→255)
pwm_linear = progress * 255
pwm_ease_in = np.array([calculate_eased_pwm(p, 0, 255, 0, 0.5) for p in progress])    # I: t 0→0.5
pwm_ease_out = np.array([calculate_eased_pwm(p, 0, 255, 0.5, 1) for p in progress])   # O: t 0.5→1
pwm_cosine = np.array([calculate_eased_pwm(p, 0, 255, 0, 1) for p in progress])       # C: t 0→1 (S-curve)

fig = go.Figure()

fig.add_trace(go.Scatter(x=time_ms/1000, y=pwm_linear, name='L (Linear)', 
                         line=dict(color='gray', width=2, dash='dash')))
fig.add_trace(go.Scatter(x=time_ms/1000, y=pwm_ease_in, name='I (Ease-In, t: 0→0.5)', 
                         line=dict(color='green', width=3)))
fig.add_trace(go.Scatter(x=time_ms/1000, y=pwm_ease_out, name='O (Ease-Out, t: 0.5→1)', 
                         line=dict(color='orange', width=3)))
fig.add_trace(go.Scatter(x=time_ms/1000, y=pwm_cosine, name='C (Cosine S-curve, t: 0→1)', 
                         line=dict(color='purple', width=3)))

fig.update_layout(
    title='PWM Intensity Over 5-Second ASCENDING Ramp (0→255)',
    xaxis_title='Time (seconds)',
    yaxis_title='PWM Value (0-255)',
    height=500,
    yaxis=dict(range=[0, 260])
)

fig.show()

## 4. X Mode vs I/O/C Modes: Key Differences

### ⚠️ CRITICAL DISTINCTION

| Feature | **I, O, C Modes** | **X Mode** |
|---------|------------------|------------|
| **Value Calculation** | Scaled: maps curve to `start → end` | Raw: `max * f(t)` directly |
| **start/end values** | **REQUIRED** - defines output range | **IGNORED** - t range defines output |
| **Protocol Format** | `(I:start,end,duration)` | `(X:duration\|t_start,t_end)` |
| **Use Case** | Scale easing shape to any value range | Direct control of cosine curve |

### Value Formats in RAMP Commands

| Value Format | Meaning | PWM Example | DAC Example |
|--------------|---------|-------------|-------------|
| `0` | OFF | 0 | 0 |
| `1` | HIGH (max) | 255 | 4095 |
| `0.5` | 50% normalized | 127 | 2047 |
| `128` | Direct 8-bit | 128 | 128 |
| `2048` | Direct 12-bit | (capped to 255) | 2048 |

### How I/O/C Modes Work (SCALED)

```
eased_progress = (f(t) - f(t_start)) / (f(t_end) - f(t_start))  # Normalize to [0,1]
output = start + eased_progress * (end - start)                  # Scale to output range
```

This means `(I:50,200,5000)` produces output from **50 to 200** with ease-in shape.

### How X Mode Works (RAW)

```
output = max_value * f(t)  # Direct curve value, start/end IGNORED!
```

The t range **directly** determines the output:
- `t: 0→0.5` → Output: **0 → 50%** (0→127 for PWM, 0→2047 for DAC)
- `t: 0→1` → Output: **0 → 100%** (0→255 for PWM, 0→4095 for DAC)
- `t: 1→2` → Output: **100% → 0**
- `t: 0→2` → Output: **0 → 100% → 0** (breathing)

### X Mode Protocol Format

```
RAMP:(X:duration|t_start,t_end)

# Examples:
RAMP:(X:5000|0,1)     # 5 seconds, 0→max
RAMP:(X:5000|1,2)     # 5 seconds, max→0
RAMP:(X:10000|0,2)    # 10 seconds breathing: 0→max→0
```

In [220]:
# Define custom t ranges to demonstrate X mode
# X MODE: Uses raw f(t) * 255 - NO SCALING
# This means the PWM directly follows the cosine curve portion

def calculate_x_mode_pwm(progress, t_start, t_end):
    """
    X mode: Raw f(t) * 255 without normalization.
    Maps progress [0,1] to t [t_start, t_end], then returns 255 * f(t).
    """
    t = t_start + progress * (t_end - t_start)
    return 255 * f(t)

custom_ranges = [
    (0, 0.5, 'Ease-In ↑ (t: 0→0.5)'),      # f: 0→0.5, PWM: 0→127.5
    (0.5, 1, 'Ease-Out ↑ (t: 0.5→1)'),     # f: 0.5→1, PWM: 127.5→255
    (1, 1.5, 'Ease-In ↓ (t: 1→1.5)'),      # f: 1→0.5, PWM: 255→127.5 (descending!)
    (1.5, 2, 'Ease-Out ↓ (t: 1.5→2)'),     # f: 0.5→0, PWM: 127.5→0 (descending!)
    (0.25, 0.75, 'Mid-range (t: 0.25→0.75)'),
]

fig = make_subplots(rows=2, cols=3, 
                    subplot_titles=[r[2] for r in custom_ranges] + ['Full curve reference'],
                    horizontal_spacing=0.08, vertical_spacing=0.15)

progress = np.linspace(0, 1, 100)
colors = px.colors.qualitative.Set2

for i, (t_start, t_end, label) in enumerate(custom_ranges):
    row = i // 3 + 1
    col = i % 3 + 1
    
    # Use raw f(t) * 255 for X mode (no scaling)
    pwm = np.array([calculate_x_mode_pwm(p, t_start, t_end) for p in progress])
    
    fig.add_trace(
        go.Scatter(x=progress, y=pwm, name=label, line=dict(color=colors[i], width=2)),
        row=row, col=col
    )
    
    # Add linear reference
    fig.add_trace(
        go.Scatter(x=progress, y=progress*255, name='Linear', 
                   line=dict(color='lightgray', dash='dash', width=1), showlegend=False),
        row=row, col=col
    )

# Add full curve reference
t_full = np.linspace(0, 2, 200)
fig.add_trace(
    go.Scatter(x=t_full, y=f(t_full)*255, name='Full curve', line=dict(color='blue', width=2)),
    row=2, col=3
)

fig.update_layout(height=600, title_text='Custom t Range Examples (Mode X) - Raw f(t) * 255', showlegend=False)
fig.update_xaxes(title_text="Progress", row=2)
fig.update_yaxes(title_text="PWM", col=1)

fig.show()

In [221]:
# Visual comparison: I/O modes (SCALED) vs X mode (RAW)
# This demonstrates the key difference!

fig = make_subplots(rows=1, cols=2, 
                    subplot_titles=['I/O/C Modes: SCALED to PWM range', 'X Mode: RAW f(t) * 255'])

progress = np.linspace(0, 1, 100)

# LEFT: I mode SCALED - same t range [0, 0.5] but different PWM ranges
for start, end, color, name in [(0, 255, 'green', 'I: 0→255'), 
                                  (50, 200, 'blue', 'I: 50→200'), 
                                  (100, 150, 'orange', 'I: 100→150')]:
    pwm = np.array([calculate_eased_pwm(p, start, end, 0, 0.5) for p in progress])
    fig.add_trace(go.Scatter(x=progress, y=pwm, name=name, 
                             line=dict(color=color, width=2)), row=1, col=1)

# RIGHT: X mode RAW - t range directly determines PWM output
for t_s, t_e, color, name in [(0, 0.5, 'green', 'X: t=0→0.5 (PWM: 0→127)'), 
                               (0.25, 0.75, 'blue', 'X: t=0.25→0.75 (PWM: 73→183)'),
                               (0.5, 1, 'orange', 'X: t=0.5→1 (PWM: 127→255)')]:
    t_values = t_s + progress * (t_e - t_s)
    pwm = 255 * f(t_values)
    fig.add_trace(go.Scatter(x=progress, y=pwm, name=name, 
                             line=dict(color=color, width=2)), row=1, col=2)

fig.update_yaxes(range=[0, 260], title_text="PWM")
fig.update_xaxes(title_text="Progress")
fig.update_layout(height=400, title_text="I/O/C Modes (SCALED) vs X Mode (RAW): The Key Difference")
fig.show()

print("KEY INSIGHT:")
print("- I/O/C modes: PWM range is controlled by start_pwm/end_pwm parameters")
print("- X mode: PWM is ALWAYS 255*f(t), the t range directly determines output")
print("\nExample: Using t=[0,0.5] in BOTH modes:")
print(f"  I mode (I:50,200,5000):  PWM goes 50 → 200 with ease-in shape")
print(f"  X mode (X:5000|0,0.5):   PWM goes 0 → 127.5 (raw curve value)")

KEY INSIGHT:
- I/O/C modes: PWM range is controlled by start_pwm/end_pwm parameters
- X mode: PWM is ALWAYS 255*f(t), the t range directly determines output

Example: Using t=[0,0.5] in BOTH modes:
  I mode (I:50,200,5000):  PWM goes 50 → 200 with ease-in shape
  X mode (X:5000|0,0.5):   PWM goes 0 → 127.5 (raw curve value)


## 5.5 Custom Easing Functions (F Mode)

### F Mode Format
```
RAMP:(F:function_name,duration_ms);
```

**Key Points:**
- **F mode only takes 2 parameters:** function name and duration
- **All functions receive `progress` ∈ [0, 1]** - time is converted to progress externally
- **Functions return output ∈ [0, 255]** - Arduino clips values after function returns
- **For DAC channels:** Arduino scales the 0-255 output to 0-4095 automatically

### Adding New Custom Functions

**Question: Is a new function immediately available after adding to `custom_easing.h`?**

| Environment | Availability |
|-------------|--------------|
| **Python Simulator** | ✅ YES - Automatically loaded when you run `--custom-funcs custom_easing.h` |
| **Protocol Parsing** | ✅ YES - Parser dynamically reads function names from header file |
| **Arduino** | ❌ NO - Requires **recompilation** and re-upload of firmware |

**Why Arduino requires recompilation:**
1. Arduino compiles C++ code at upload time
2. New function names need to be added to a dispatch table in the firmware
3. The firmware must be modified to call your new function based on its name

**To add a new function:**
1. Add to `light_controller_v2_3_arduino/custom_easing.h`:
```c
// CUSTOM_FUNC: my_new_func
// PYTHON_EQUIV: lambda p: 255 * (p ** 2)

float my_new_func(float progress) {
    return 255.0 * progress * progress;  // Quadratic
}
```

2. For Python simulator: Works immediately!
3. For Arduino: Modify firmware to handle the new function name in the dispatch logic

### All Available Custom Functions

In [ ]:
# Visualize ALL custom functions from custom_easing.h
import re
import math
from pathlib import Path

# Load custom functions from header file (updated path for v2.3)
header_path = Path("../light_controller_v2_3_arduino/custom_easing.h")

custom_functions = {}
if header_path.exists():
    content = header_path.read_text()
    
    # Parse CUSTOM_FUNC and PYTHON_EQUIV pairs
    pattern = r'//\s*CUSTOM_FUNC:\s*(\w+)\s*\n\s*//\s*PYTHON_EQUIV:\s*(.+)'
    matches = re.findall(pattern, content)
    
    for name, expr in matches:
        try:
            func = eval(expr, {'math': math, 'sin': math.sin, 'cos': math.cos, 
                               'exp': math.exp, 'log': math.log, 'sqrt': math.sqrt,
                               'abs': abs, 'min': min, 'max': max,
                               'pi': math.pi, 'PI': math.pi, 'e': math.e})
            custom_functions[name] = func
        except Exception as e:
            print(f"Failed to load {name}: {e}")
    
    print(f"✓ Loaded {len(custom_functions)} custom functions from {header_path.name}:")
    for name in custom_functions:
        print(f"  • {name}")
else:
    print(f"⚠️ Header file not found: {header_path}")
    print("  Creating sample functions for demonstration...")
    # Fallback sample functions
    custom_functions = {
        'sine_wave': lambda p: 255 * math.sin(p * math.pi),
        'heartbeat': lambda p: min(255, 255 * (math.exp(-20*(p-0.2)**2) + 0.6*math.exp(-20*(p-0.4)**2))),
        'bounce': lambda p: 255 * abs(math.sin(p * math.pi * 3) * (1 - p)),
        'breathing': lambda p: 127.5 * (1 - math.cos(2 * math.pi * p)),
    }

# Create visualization grid
n_funcs = len(custom_functions)
cols = 3
rows = (n_funcs + cols - 1) // cols

fig = make_subplots(
    rows=rows, cols=cols,
    subplot_titles=list(custom_functions.keys()),
    vertical_spacing=0.08,
    horizontal_spacing=0.08
)

progress = np.linspace(0, 1, 200)
colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6', '#f39c12', 
          '#1abc9c', '#e91e63', '#00bcd4', '#ff5722', '#795548']

for idx, (name, func) in enumerate(custom_functions.items()):
    row = idx // cols + 1
    col = idx % cols + 1
    
    try:
        pwm = np.array([func(p) for p in progress])
        # Clip for display (Arduino clips after function)
        pwm = np.clip(pwm, 0, 255)
    except Exception as e:
        pwm = np.zeros_like(progress)
        print(f"Error evaluating {name}: {e}")
    
    fig.add_trace(
        go.Scatter(
            x=progress, y=pwm,
            mode='lines',
            name=name,
            line=dict(color=colors[idx % len(colors)], width=2),
            showlegend=False
        ),
        row=row, col=col
    )
    
    # Add axis labels
    fig.update_xaxes(title_text="Progress [0,1]", row=row, col=col)
    fig.update_yaxes(title_text="Output", range=[0, 270], row=row, col=col)

fig.update_layout(
    title=dict(text="Custom Easing Functions from custom_easing.h", x=0.5),
    height=300 * rows,
    template="plotly_white"
)

fig.show()

# Print protocol usage examples
print("\n" + "=" * 50)
print("Protocol Usage Examples:")
print("=" * 50)
for name in list(custom_functions.keys())[:5]:
    print(f"PATTERN:1;CH:1;RAMP:(F:{name},5000);REPEATS:3")
print("...")

Loaded 11 custom functions from custom_easing.h:
  • sine_wave
  • double_sine
  • bounce
  • heartbeat
  • exp_decay
  • log_rise
  • step_50
  • sawtooth
  • breathing
  • flicker
  • myfunc



Protocol Usage Examples:
PATTERN:1;CH:1;RAMP:(F:sine_wave,5000);REPEATS:3
PATTERN:1;CH:1;RAMP:(F:double_sine,5000);REPEATS:3
PATTERN:1;CH:1;RAMP:(F:bounce,5000);REPEATS:3
PATTERN:1;CH:1;RAMP:(F:heartbeat,5000);REPEATS:3
PATTERN:1;CH:1;RAMP:(F:exp_decay,5000);REPEATS:3
...


## 5. Breathing Effect (Full Cycle t: 0→2)

Using `X` mode with `t: 0→2` creates a complete breathing cycle in a single segment:

In [223]:
# Breathing effect: 6 second cycle, repeated 3 times
cycle_ms = 6000
cycles = 3
total_ms = cycle_ms * cycles

time_ms = np.linspace(0, total_ms, 500)

# Calculate PWM using f(t) with t: 0→2 for each cycle
pwm_breathing = []
for t in time_ms:
    cycle_progress = (t % cycle_ms) / cycle_ms  # 0 to 1 within each cycle
    t_param = cycle_progress * 2  # Map to 0 to 2
    pwm = f(t_param) * 255  # Direct use of f(t) since we want min→max→min
    pwm_breathing.append(pwm)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=time_ms/1000, y=pwm_breathing,
    mode='lines',
    name='Breathing (X mode, t: 0→2)',
    line=dict(color='purple', width=3),
    fill='tozeroy',
    fillcolor='rgba(128, 0, 128, 0.2)'
))

# Add cycle markers
for i in range(cycles + 1):
    fig.add_vline(x=i * cycle_ms / 1000, line=dict(color='gray', dash='dash', width=1))

fig.update_layout(
    title='Breathing Effect: RAMP:(X:0,255,6000|0,2) repeated 3 times',
    xaxis_title='Time (seconds)',
    yaxis_title='PWM Intensity',
    height=400,
    yaxis=dict(range=[0, 260])
)

fig.show()

## 6. Multi-Segment RAMP Example

Combining multiple segments with different easing modes:

In [224]:
# Multi-segment RAMP: fade-in (ease-in), hold, fade-out (ease-out)
# (I:0,255,3000),(L:255,255,2000),(O:255,0,3000)
# Note: Ease-in ASCENDING uses t: 0→0.5, Ease-out DESCENDING uses t: 1.5→2

segments = [
    {'mode': 'I', 'start': 0, 'end': 255, 'duration': 3000, 't_start': 0, 't_end': 0.5, 'label': 'Ease-In ↑'},
    {'mode': 'L', 'start': 255, 'end': 255, 'duration': 2000, 't_start': 0, 't_end': 1, 'label': 'Hold'},
    {'mode': 'O', 'start': 255, 'end': 0, 'duration': 3000, 't_start': 1.5, 't_end': 2, 'label': 'Ease-Out ↓'},
]

time_all = []
pwm_all = []
colors_seg = ['green', 'gray', 'orange']
current_time = 0

fig = go.Figure()

for i, seg in enumerate(segments):
    n_points = seg['duration'] // 20
    time_seg = np.linspace(current_time, current_time + seg['duration'], n_points)
    progress = np.linspace(0, 1, n_points)
    
    if seg['mode'] == 'L':
        pwm = np.full(n_points, seg['start'])
    else:
        pwm = np.array([calculate_eased_pwm(p, seg['start'], seg['end'], 
                                            seg['t_start'], seg['t_end']) for p in progress])
    
    fig.add_trace(go.Scatter(
        x=time_seg/1000, y=pwm,
        mode='lines',
        name=f"{seg['label']} ({seg['mode']})",
        line=dict(color=colors_seg[i], width=3)
    ))
    
    current_time += seg['duration']

fig.update_layout(
    title='Multi-Segment RAMP: (I:0,255,3000) → Hold → (O:255,0,3000)',
    xaxis_title='Time (seconds)',
    yaxis_title='PWM Intensity',
    height=400,
    yaxis=dict(range=[0, 260])
)

fig.show()

## 7. t Range Selection Guide

Interactive visualization showing how different t ranges affect the output:

In [225]:
# Create a comprehensive t-range selection guide
t_starts = [0, 0, 0.5, 1, 1.5]
t_ends = [0.5, 1, 1, 1.5, 2]
labels = ['Very slow start', 'Ease-In (standard)', 'Accelerating', 'Decelerating', 'Very slow end']

progress = np.linspace(0, 1, 100)
colors = ['#2ecc71', '#3498db', '#9b59b6', '#e74c3c', '#f39c12']

fig = make_subplots(rows=1, cols=2, 
                    subplot_titles=['Normalized Progress', 'Actual PWM (0→255)'],
                    horizontal_spacing=0.1)

for i, (t_s, t_e, label) in enumerate(zip(t_starts, t_ends, labels)):
    # Normalized progress (0-1)
    eased = np.array([calculate_eased_pwm(p, 0, 1, t_s, t_e) for p in progress])
    
    fig.add_trace(
        go.Scatter(x=progress, y=eased, name=f'{label} (t:{t_s}→{t_e})',
                   line=dict(color=colors[i], width=2)),
        row=1, col=1
    )
    
    # PWM values (0-255)
    pwm = np.array([calculate_eased_pwm(p, 0, 255, t_s, t_e) for p in progress])
    
    fig.add_trace(
        go.Scatter(x=progress, y=pwm, name=f'{label}',
                   line=dict(color=colors[i], width=2), showlegend=False),
        row=1, col=2
    )

# Add linear reference
fig.add_trace(
    go.Scatter(x=progress, y=progress, name='Linear (L)',
               line=dict(color='gray', dash='dash', width=1)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=progress, y=progress*255, name='Linear',
               line=dict(color='gray', dash='dash', width=1), showlegend=False),
    row=1, col=2
)

fig.update_layout(
    title='t Range Selection Guide',
    height=450,
    legend=dict(x=0.02, y=0.98)
)
fig.update_xaxes(title_text="Time Progress (0-1)")
fig.update_yaxes(title_text="Eased Progress", row=1, col=1)
fig.update_yaxes(title_text="PWM Value", row=1, col=2)

fig.show()

## 8. Command Format Reference

### Value Format Reference

| Input | PWM (0-255) | DAC (0-4095) | Description |
|-------|-------------|--------------|-------------|
| `0` | 0 | 0 | OFF (minimum) |
| `1` (integer) | **255** | **4095** | HIGH (maximum) |
| `0.0` | 0 | 0 | Normalized OFF |
| `1.0` (float) | **255** | **4095** | Normalized HIGH |
| `0.5` | 127 | 2047 | 50% brightness |
| `0.25` | 63 | 1023 | 25% brightness |
| `100` | 100 | 100 | Direct value |
| `2000` | - | 2000 | DAC direct (>255) |

> ⚠️ **Key Point:** Both `STATUS:1` (int) and `STATUS:1.0` (float) are treated as HIGH!

### New Parenthesized RAMP Format

```
RAMP:(<MODE>:<start>,<end>,<duration>[,<steps>][|<t_start>,<t_end>])
```

**Examples:**

| Command | Description |
|---------|-------------|
| `RAMP:(L:0,255,5000)` | Linear ramp 0→255 over 5s |
| `RAMP:(I:0,255,5000)` | Ease-in ramp (slow start) |
| `RAMP:(O:255,0,5000)` | Ease-out ramp (slow end) |
| `RAMP:(C:0,255,5000)` | Ease-in-out (smooth S-curve) |
| `RAMP:(X:0,255,5000\|0,2)` | Custom t-range cycle |
| `RAMP:(I:0,1,3000),(L:1,1,2000),(O:1,0,3000)` | Multi-segment with normalized values |
| `RAMP:(F:sine_wave,5000)` | Custom function over 5s |

### t Range Quick Reference

| t Range | Effect | Use Case |
|---------|--------|----------|
| `0,1` | Ease-in (slow start) | Sunrise, fade-in |
| `1,2` | Ease-out (slow end) | Sunset, fade-out |
| `0,2` | Full cycle (0→1→0) | Breathing, pulse |
| `0,0.5` | Very gradual start | Gentle wake-up |
| `1.5,2` | Very gradual end | Gentle sleep |

In [226]:
# Print summary table of f(t) values at key points
print("f(t) = (1 - cos(πt)) / 2 at key points:")
print("="*40)
print(f"{'t':>6} | {'f(t)':>8} | {'PWM (0-255)':>12}")
print("-"*40)
for t in [0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0]:
    ft = f(t)
    pwm = int(ft * 255)
    print(f"{t:>6.2f} | {ft:>8.4f} | {pwm:>12}")

f(t) = (1 - cos(πt)) / 2 at key points:
     t |     f(t) |  PWM (0-255)
----------------------------------------
  0.00 |   0.0000 |            0
  0.25 |   0.1464 |           37
  0.50 |   0.5000 |          127
  0.75 |   0.8536 |          217
  1.00 |   1.0000 |          255
  1.25 |   0.8536 |          217
  1.50 |   0.5000 |          127
  1.75 |   0.1464 |           37
  2.00 |   0.0000 |            0


## 9. Protocol Examples

### Text Protocol Format (.txt)

The text protocol format uses PATTERN commands with semicolon separators:

```
# Standard PATTERN commands
PATTERN:<id>;CH:<channel>;RAMP:<ramp_spec>;REPEATS:<count>
PATTERN:<id>;CH:<channel>;STATUS:<value>;TIME_MS:<duration>;REPEATS:<count>

# STATUS values: 0, 1, 0.0-1.0, or direct values (2-255 for PWM, 2-4095 for DAC)
PATTERN:1;CH:1;STATUS:1;TIME_MS:5000;REPEATS:1       # HIGH (255 or 4095)
PATTERN:2;CH:1;STATUS:0.5;TIME_MS:5000;REPEATS:1     # 50% (127 or 2047)
PATTERN:3;CH:1;STATUS:100;TIME_MS:5000;REPEATS:1     # Direct value 100

# LOOP syntax for repeating sections
LOOP:START:<count>
PATTERN:...
LOOP:END
```

### Excel Protocol Format

Excel files have columns for each channel:
- `CH<n>_status`: Pattern status (`0`, `1`, `0.0-1.0`, direct values, or `"ramp"`)
- `CH<n>_time_sec` or `CH<n>_time_ms`: Duration
- `CH<n>_ramp`: Ramp specification (for ramp rows)

In [227]:

# Example 1: Sunrise Simulation Protocol
sunrise_protocol = """
# ===================================================================================
# SUNRISE SIMULATION - Natural light progression
# ===================================================================================

# Phase 1: Gentle ease-in sunrise over 30 minutes (slow start like real sunrise)
PATTERN:1;CH:1;RAMP:(I:0,255,1800000);REPEATS:1

# Phase 2: Hold at full brightness for 2 hours
PATTERN:2;CH:1;STATUS:255;TIME_MS:7200000;REPEATS:1

# Phase 3: Gradual sunset over 15 minutes (slow end like real sunset)  
PATTERN:3;CH:1;RAMP:(O:255,0,900000);REPEATS:1

START_TIME: {'CH1': 30}
"""

print("Sunrise Protocol:")
print(sunrise_protocol)


Sunrise Protocol:

# ===================================================================================
# SUNRISE SIMULATION - Natural light progression
# ===================================================================================

# Phase 1: Gentle ease-in sunrise over 30 minutes (slow start like real sunrise)
PATTERN:1;CH:1;RAMP:(I:0,255,1800000);REPEATS:1

# Phase 2: Hold at full brightness for 2 hours
PATTERN:2;CH:1;STATUS:255;TIME_MS:7200000;REPEATS:1

# Phase 3: Gradual sunset over 15 minutes (slow end like real sunset)  
PATTERN:3;CH:1;RAMP:(O:255,0,900000);REPEATS:1

START_TIME: {'CH1': 30}



In [228]:
# Visualize Protocol Timelines with correct easing
def plot_protocol_timeline(patterns, channel_num, title):
    """Plot a protocol timeline with all segments using correct easing"""
    all_times = []
    all_pwms = []
    current_time = 0
    
    for pattern in patterns:
        duration = pattern['duration_ms']
        segments = pattern.get('segments', [])
        
        if not segments:
            # Simple constant value (hold)
            pwm = pattern.get('pwm', 0)
            all_times.extend([current_time / 1000, (current_time + duration) / 1000])
            all_pwms.extend([pwm, pwm])
            current_time += duration
        else:
            # Multi-segment ramp
            for seg in segments:
                seg_duration = seg['duration']
                t_start = seg.get('t_start', 0)
                t_end = seg.get('t_end', 0.5)
                start_pwm = seg['start']
                end_pwm = seg['end']
                
                # Generate points for this segment
                num_points = max(50, seg_duration // 1000)  # At least 50 points
                for i in range(num_points + 1):
                    progress = i / num_points
                    time_ms = current_time + progress * seg_duration
                    pwm = calculate_eased_pwm(progress, start_pwm, end_pwm, t_start, t_end)
                    
                    all_times.append(time_ms / 1000)  # Convert to seconds
                    all_pwms.append(pwm)
                
                current_time += seg_duration
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=all_times, y=all_pwms,
        mode='lines',
        fill='tozeroy',
        name=f'Channel {channel_num}',
        fillcolor='rgba(102, 126, 234, 0.3)',
        line=dict(color='#667eea', width=2)
    ))
    
    fig.update_layout(
        title=title,
        xaxis_title='Time (seconds)',
        yaxis_title='PWM Value (0-255)',
        template='plotly_white',
        hovermode='x unified'
    )
    
    return fig

# =============================================================================
# SUNRISE SIMULATION PROTOCOL
# =============================================================================
# Phase 1: Ease-in sunrise (slow start, accelerating) - t=[0, 0.5]
# Phase 2: Hold at full brightness
# Phase 3: Ease-out sunset (fast start, decelerating) - t=[1.5, 2]

sunrise_segments = [
    {
        'duration_ms': 60000,  # 1 minute for demo (scale up for real: 1800000 = 30 min)
        'segments': [{'start': 0, 'end': 255, 'duration': 60000, 't_start': 0, 't_end': 0.5}]  # Ease-in (ascending)
    },
    {
        'duration_ms': 120000,  # 2 minutes hold (scale up: 7200000 = 2 hours)
        'pwm': 255
    },
    {
        'duration_ms': 30000,  # 30 seconds sunset (scale up: 900000 = 15 min)
        'segments': [{'start': 255, 'end': 0, 'duration': 30000, 't_start': 1.5, 't_end': 2}]  # Ease-out (descending)
    }
]

fig_sunrise = plot_protocol_timeline(sunrise_segments, 1, "Sunrise Simulation (Ease-in→Hold→Ease-out)")
fig_sunrise.show()

In [229]:

# Example 2: Breathing/Circadian Rhythm Protocol (Complex Multi-Phase)
breathing_protocol = """
# ===================================================================================
# CIRCADIAN RHYTHM - Mimics natural day/night cycle with breathing
# ===================================================================================

# Night Phase: Sleep mode with subtle breathing (25% min, slow pulse)
PATTERN:1;CH:1;RAMP:(X:25,75,4000|0,1),(X:75,25,4000|1,2);REPEATS:12

# Dawn Phase: Gradual awakening (10 minutes)
PATTERN:2;CH:1;RAMP:(I:25,180,600000);REPEATS:1

# Day Phase: Full brightness steady
PATTERN:3;CH:1;STATUS:255;TIME_MS:43200000;REPEATS:1

# Dusk Phase: Gradual dimming (10 minutes)
PATTERN:4;CH:1;RAMP:(O:180,25,600000);REPEATS:1

# Night Breathing: Return to sleep mode
PATTERN:5;CH:1;RAMP:(X:25,75,4000|0,1),(X:75,25,4000|1,2);REPEATS:12

START_TIME: {'CH1': 0}
"""

print("Circadian Rhythm Protocol:")
print(breathing_protocol)


Circadian Rhythm Protocol:

# ===================================================================================
# CIRCADIAN RHYTHM - Mimics natural day/night cycle with breathing
# ===================================================================================

# Night Phase: Sleep mode with subtle breathing (25% min, slow pulse)
PATTERN:1;CH:1;RAMP:(X:25,75,4000|0,1),(X:75,25,4000|1,2);REPEATS:12

# Dawn Phase: Gradual awakening (10 minutes)
PATTERN:2;CH:1;RAMP:(I:25,180,600000);REPEATS:1

# Day Phase: Full brightness steady
PATTERN:3;CH:1;STATUS:255;TIME_MS:43200000;REPEATS:1

# Dusk Phase: Gradual dimming (10 minutes)
PATTERN:4;CH:1;RAMP:(O:180,25,600000);REPEATS:1

# Night Breathing: Return to sleep mode
PATTERN:5;CH:1;RAMP:(X:25,75,4000|0,1),(X:75,25,4000|1,2);REPEATS:12

START_TIME: {'CH1': 0}



In [230]:
# Visualize Circadian Rhythm with CORRECT t ranges
# Breathing: up with ease-in [0, 0.5], down with ease-out [1.5, 2]
# Dawn: ease-in ascending [0, 0.5]  
# Dusk: ease-out descending [1.5, 2]

circadian_segments = [
    # Night breathing cycle (slow pulse: ease-in up, ease-out down)
    {'duration_ms': 8000, 'segments': [
        {'start': 25, 'end': 75, 'duration': 4000, 't_start': 0, 't_end': 0.5},   # Ease-in UP
        {'start': 75, 'end': 25, 'duration': 4000, 't_start': 1.5, 't_end': 2}    # Ease-out DOWN
    ]},
    # Dawn: ease-in ascending (slow start)
    {'duration_ms': 30000, 'segments': [
        {'start': 25, 'end': 180, 'duration': 30000, 't_start': 0, 't_end': 0.5}  # Ease-in UP
    ]},
    # Day: hold at full brightness
    {'duration_ms': 60000, 'pwm': 255},
    # Dusk: ease-out descending (slow end)
    {'duration_ms': 30000, 'segments': [
        {'start': 180, 'end': 25, 'duration': 30000, 't_start': 1.5, 't_end': 2}  # Ease-out DOWN
    ]},
    # Night breathing again
    {'duration_ms': 8000, 'segments': [
        {'start': 25, 'end': 75, 'duration': 4000, 't_start': 0, 't_end': 0.5},
        {'start': 75, 'end': 25, 'duration': 4000, 't_start': 1.5, 't_end': 2}
    ]}
]

fig_circadian = plot_protocol_timeline(circadian_segments, 1, "Circadian Rhythm (Breathing→Dawn→Day→Dusk→Breathing)")
fig_circadian.show()

In [231]:

# Example 3: Multi-Channel Experimental Protocol
multi_channel_protocol = """
# ===================================================================================
# MULTI-CHANNEL EXPERIMENT - Staggered light sequences
# ===================================================================================

# Channel 1: Linear brightness progression
PATTERN:1;CH:1;RAMP:(L:0,255,5000);REPEATS:2

# Channel 2: Ease-in progression (slower start)
PATTERN:1;CH:2;RAMP:(I:0,255,5000);REPEATS:2

# Channel 3: Ease-out progression (slower end)
PATTERN:1;CH:3;RAMP:(O:0,255,5000);REPEATS:2

# Channel 4: Cosine smooth S-curve
PATTERN:1;CH:4;RAMP:(C:0,255,5000);REPEATS:2

# Combine all in a loop with holds
PATTERN:2;CH:1;STATUS:255,0;TIME_MS:1000,1000;REPEATS:3
PATTERN:2;CH:2;STATUS:255,0;TIME_MS:1000,1000;REPEATS:3
PATTERN:2;CH:3;STATUS:255,0;TIME_MS:1000,1000;REPEATS:3
PATTERN:2;CH:4;STATUS:255,0;TIME_MS:1000,1000;REPEATS:3
"""

print("Multi-Channel Protocol:")
print(multi_channel_protocol)


Multi-Channel Protocol:

# ===================================================================================
# MULTI-CHANNEL EXPERIMENT - Staggered light sequences
# ===================================================================================

# Channel 1: Linear brightness progression
PATTERN:1;CH:1;RAMP:(L:0,255,5000);REPEATS:2

# Channel 2: Ease-in progression (slower start)
PATTERN:1;CH:2;RAMP:(I:0,255,5000);REPEATS:2

# Channel 3: Ease-out progression (slower end)
PATTERN:1;CH:3;RAMP:(O:0,255,5000);REPEATS:2

# Channel 4: Cosine smooth S-curve
PATTERN:1;CH:4;RAMP:(C:0,255,5000);REPEATS:2

# Combine all in a loop with holds
PATTERN:2;CH:1;STATUS:255,0;TIME_MS:1000,1000;REPEATS:3
PATTERN:2;CH:2;STATUS:255,0;TIME_MS:1000,1000;REPEATS:3
PATTERN:2;CH:3;STATUS:255,0;TIME_MS:1000,1000;REPEATS:3
PATTERN:2;CH:4;STATUS:255,0;TIME_MS:1000,1000;REPEATS:3



In [232]:
# Visualize all 4 easing modes side-by-side with CORRECT t ranges
fig_all_modes = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Linear (L)', 'Ease-In (I) t:0→0.5', 'Ease-Out (O) t:0.5→1', 'Cosine (C) t:0→1')
)

# Correct modes with proper t ranges for ASCENDING
modes = [
    ('Linear', 0, 1, 'L'),      # Linear doesn't use easing
    ('Ease-In', 0, 0.5, 'I'),   # Slow start for ascending
    ('Ease-Out', 0.5, 1, 'O'),  # Slow end for ascending
    ('Cosine', 0, 1, 'C')       # Full S-curve
]

colors = ['gray', 'green', 'orange', 'purple']

for idx, (mode_name, t_start, t_end, mode_code) in enumerate(modes):
    row = idx // 2 + 1
    col = idx % 2 + 1
    
    times = []
    pwms = []
    
    num_points = 100
    for i in range(num_points + 1):
        progress = i / num_points
        if mode_code == 'L':
            pwm = progress * 255  # Linear
        else:
            pwm = calculate_eased_pwm(progress, 0, 255, t_start, t_end)
        times.append(progress * 5)  # Scale to 5 seconds
        pwms.append(pwm)
    
    fig_all_modes.add_trace(
        go.Scatter(x=times, y=pwms, mode='lines', 
                   fill='tozeroy', name=mode_name,
                   line=dict(width=2, color=colors[idx])),
        row=row, col=col
    )

fig_all_modes.update_yaxes(range=[0, 260])
fig_all_modes.update_layout(height=600, showlegend=False, title_text="All Easing Modes - ASCENDING (0→255)")
fig_all_modes.show()

## 6. Combined Protocol: Ramps + Constant Periods

Real-world protocols often combine ramped transitions with constant ON/OFF periods. Here's a comprehensive example showing all combinations:

In [233]:
# =============================================================================
# COMBINED PROTOCOL: Ramps + Constant ON/OFF + Multiple Segments
# =============================================================================
# This demonstrates a realistic laboratory lighting protocol

combined_protocol = """
# Laboratory Experiment Protocol: 
# OFF → Ease-in ON → HOLD ON → Pulse (2x) → HOLD ON → Ease-out OFF → HOLD OFF

PATTERN:1;CH:1;RAMP:(I:0,200,5000);STATUS:200;TIME_MS:10000;RAMP:(O:200,0,3000);STATUS:0;TIME_MS:5000;REPEATS:1
"""

# Visualize the combined protocol
combined_segments = [
    # Phase 1: OFF period
    {'duration_ms': 3000, 'pwm': 0},
    
    # Phase 2: Ease-in ramp (slow start, accelerating)
    {'duration_ms': 5000, 'segments': [
        {'start': 0, 'end': 200, 'duration': 5000, 't_start': 0, 't_end': 0.5}  # Ease-in UP
    ]},
    
    # Phase 3: HOLD at 200
    {'duration_ms': 10000, 'pwm': 200},
    
    # Phase 4: Pulse sequence (2 fast pulses)
    {'duration_ms': 500, 'segments': [
        {'start': 200, 'end': 255, 'duration': 250, 't_start': 0, 't_end': 0.5},  # Quick up
        {'start': 255, 'end': 200, 'duration': 250, 't_start': 1.5, 't_end': 2}   # Quick down
    ]},
    {'duration_ms': 500, 'pwm': 200},  # Brief hold
    {'duration_ms': 500, 'segments': [
        {'start': 200, 'end': 255, 'duration': 250, 't_start': 0, 't_end': 0.5},
        {'start': 255, 'end': 200, 'duration': 250, 't_start': 1.5, 't_end': 2}
    ]},
    
    # Phase 5: HOLD at 200
    {'duration_ms': 8000, 'pwm': 200},
    
    # Phase 6: Ease-out ramp (fast start, slow end - gentle fade)
    {'duration_ms': 5000, 'segments': [
        {'start': 200, 'end': 0, 'duration': 5000, 't_start': 1.5, 't_end': 2}  # Ease-out DOWN
    ]},
    
    # Phase 7: HOLD OFF
    {'duration_ms': 5000, 'pwm': 0}
]

fig_combined = plot_protocol_timeline(combined_segments, 1, "Combined Protocol: OFF→Ease-in→HOLD→Pulse→HOLD→Ease-out→OFF")

# Add phase annotations
phases = [
    (1.5, "OFF"), (5.5, "Ease-in"), (13, "HOLD"), 
    (19, "Pulse"), (24, "HOLD"), (29, "Ease-out"), (35, "OFF")
]
for x, label in phases:
    fig_combined.add_annotation(x=x, y=240, text=label, showarrow=False, 
                                 font=dict(size=10, color="gray"))

fig_combined.show()

print("Protocol Text Format:")
print(combined_protocol)

Protocol Text Format:

# Laboratory Experiment Protocol: 
# OFF → Ease-in ON → HOLD ON → Pulse (2x) → HOLD ON → Ease-out OFF → HOLD OFF

PATTERN:1;CH:1;RAMP:(I:0,200,5000);STATUS:200;TIME_MS:10000;RAMP:(O:200,0,3000);STATUS:0;TIME_MS:5000;REPEATS:1



## 10. Real-World Protocol Examples

### Protocol Example Files

> 📂 **View real protocol examples in the [examples/](../examples/) folder**

#### Text Protocol Files (.txt)
| File | Description |
|------|-------------|
| [clean_protocol.txt](../examples/clean_protocol.txt) | Minimal protocol example |
| [complete_protocol.txt](../examples/complete_protocol.txt) | Full-featured protocol demo |
| [1min_test.txt](../examples/1min_test.txt) | Quick 1-minute test |
| [5min_pwm_ramp_demo.txt](../examples/5min_pwm_ramp_demo.txt) | 5-minute PWM ramp demo |
| [dac_output_demo.txt](../examples/dac_output_demo.txt) | DAC channel (0-4095) examples |
| [mixed_channel_types.txt](../examples/mixed_channel_types.txt) | Mixed PWM/DAC channels |

#### Ramp Easing Examples (examples/ramp_easing/)
| File | Description |
|------|-------------|
| [comprehensive_easing_modes.txt](../examples/ramp_easing/comprehensive_easing_modes.txt) | All easing modes (L,I,O,C,X) |
| [f_mode_demo.txt](../examples/ramp_easing/f_mode_demo.txt) | Custom F-mode functions |
| [x_mode_demo.txt](../examples/ramp_easing/x_mode_demo.txt) | X-mode custom t-range examples |
| [pwm_ramp_protocol.txt](../examples/ramp_easing/pwm_ramp_protocol.txt) | PWM ramp control examples |

#### HTML Visualizations (examples/sample_outputs/)
| File | Description |
|------|-------------|
| [pwm_ramp_protocol.html](../examples/sample_outputs/pwm_ramp_protocol.html) | Interactive PWM visualization |
| [test_ramp_visualization.html](../examples/sample_outputs/test_ramp_visualization.html) | Ramp testing visualization |
| [5min_pwm_ramp_demo_simulation.html](../examples/sample_outputs/5min_pwm_ramp_demo_simulation.html) | 5-minute demo visualization |

---

### Value Format Reference

| Input Format | PWM Channel (P) | DAC Channel (D) | Description |
|--------------|-----------------|-----------------|-------------|
| `0` | 0 | 0 | OFF (minimum) |
| `1` (int) | 255 | 4095 | ON (maximum) |
| `0.0` - `1.0` | 0-255 (scaled) | 0-4095 (scaled) | Normalized percentage |
| `2` - `255` | Direct value | - | PWM direct value |
| `2` - `4095` | - | Direct value | DAC direct value |

---

### Documentation Links

> 📚 **Complete documentation in the [docs/](.) folder**

#### Core Documentation
| Document | Description |
|----------|-------------|
| [PROTOCOL_FORMATS.md](PROTOCOL_FORMATS.md) | **Protocol syntax reference** |
| [PROTOCOL_SYNTAX_REFERENCE.md](PROTOCOL_SYNTAX_REFERENCE.md) | Detailed syntax rules |
| [PWM_RAMP_CONTROL.md](PWM_RAMP_CONTROL.md) | **Easing modes & RAMP guide** |
| [F_MODE_CUSTOM_FUNCTIONS.md](F_MODE_CUSTOM_FUNCTIONS.md) | Custom function implementation |
| [USAGE.md](USAGE.md) | Command line usage |
| [FEATURES.md](FEATURES.md) | Feature overview |

#### Visualization & Preview
| Document | Description |
|----------|-------------|
| [HTML_VISUALIZATION.md](HTML_VISUALIZATION.md) | HTML output guide |
| [VISUALIZATION_GUIDE.md](VISUALIZATION_GUIDE.md) | Visualization overview |
| [PREVIEW_GUIDE.md](PREVIEW_GUIDE.md) | Protocol preview features |

#### Calibration & Setup
| Document | Description |
|----------|-------------|
| [CALIBRATION_GUIDE.md](CALIBRATION_GUIDE.md) | Calibration procedures |
| [AUTO_CALIBRATION_DATABASE.md](AUTO_CALIBRATION_DATABASE.md) | Auto-calibration database |
| [ARDUINO_SETUP.md](ARDUINO_SETUP.md) | Arduino configuration |
| [INSTALLATION.md](INSTALLATION.md) | Installation guide |

#### Advanced Topics
| Document | Description |
|----------|-------------|
| [PULSE_MODE_TESTING_GUIDE.md](PULSE_MODE_TESTING_GUIDE.md) | Pulse mode testing |
| [PATTERN_COMPRESSION_GUIDE.md](PATTERN_COMPRESSION_GUIDE.md) | Pattern optimization |
| [TROUBLESHOOTING.md](TROUBLESHOOTING.md) | Common issues & solutions |

---

### Text Protocol Quick Examples

You can copy these examples directly into `.txt` protocol files:

**Example 1: Simple ON/OFF with normalized values**
```
# Using 1 (int) as HIGH - outputs 255 for PWM, 4095 for DAC
PATTERN:1;CH:1;STATUS:1;TIME_MS:5000;REPEATS:1
# Using 0.5 (float) as 50% - outputs 127 for PWM, 2047 for DAC
PATTERN:2;CH:1;STATUS:0.5;TIME_MS:5000;REPEATS:1
```

**Example 2: Sunrise Ramp (5 minutes)**
```
PATTERN:1;CH:1;RAMP:(I:0,255,300000);REPEATS:1
PATTERN:2;CH:1;STATUS:1;TIME_MS:600000;REPEATS:1
```

**Example 3: Breathing Loop with X-mode (30 seconds)**
```
PATTERN:1;CH:1;RAMP:(X:50,200,15000|0,1),(X:200,50,15000|1,2);REPEATS:10
```

**Example 4: Full Day Cycle with LOOP**
```
LOOP:START:24
PATTERN:1;CH:1;RAMP:(I:0,1,1800000);REPEATS:1
PATTERN:2;CH:1;STATUS:1.0;TIME_MS:3600000;REPEATS:1
PATTERN:3;CH:1;RAMP:(O:1,0,900000);REPEATS:1
PATTERN:4;CH:1;STATUS:0;TIME_MS:3600000;REPEATS:1
LOOP:END
```

> **Note:** `STATUS:1` and `STATUS:1.0` are both treated as maximum (255 for PWM, 4095 for DAC)